In [1]:
import duckdb

In [2]:
import os
os.listdir('D:/akiml_data/vitaldb')

['.ipynb_checkpoints',
 '01-vitaldb-explore.ipynb',
 'clinical_data.csv',
 'clinical_parameters.csv',
 'lab_data.csv',
 'lab_parameters.csv',
 'track_names.csv',
 'Untitled.ipynb']

In [3]:
duckdb.sql("SELECT COUNT(*) AS n FROM 'D:/akiml_data/vitaldb/clinical_data.csv'").df()

,n
0,6388


In [4]:
duckdb.sql("SELECT * FROM 'D:/akiml_data/vitaldb/lab_parameters.csv'").df()

,Parameter,Category,Description,Unit,Reference value
0,wbc,CBC,White blood cell count,×1000/mcL,4~10
1,hb,CBC,Hemoglobin,g/dL,13~17
2,hct,CBC,Hematocrit,%,39~52
3,plt,CBC,Platelet count,×1000/mcL,130~400
4,esr,CBC,Erythrocyte sedimentation rate,mm/hr,0~9
5,gluc,Chemistry,Glucose,mg/dL,70~110
6,tprot,Chemistry,Total protein,g/dL,6.0~8.0
7,alb,Chemistry,Albumin,g/dL,3.3~5.2
8,tbil,Chemistry,Total bilirubin,mg/dL,0.2~1.2
9,ast,Chemistry,Asparate transferase,IU/L,1~40


In [5]:
duckdb.sql("SELECT * FROM 'D:/akiml_data/vitaldb/lab_data.csv' LIMIT 5").df()

,caseid,dt,name,result
0,1,594470,alb,2.9
1,1,399575,alb,3.2
2,1,12614,alb,3.4
3,1,137855,alb,3.6
4,1,399575,alt,12.0


In [6]:
duckdb.sql("SELECT COUNT(*) AS n_cre FROM 'D:/akiml_data/vitaldb/lab_data.csv' WHERE name = 'CRE'").df()

,n_cre
0,0


In [7]:
duckdb.sql("SELECT * FROM 'D:/akiml_data/vitaldb/clinical_data.csv' LIMIT 3").df()

,caseid,subjectid,casestart,caseend,anestart,aneend,opstart,opend,adm,dis,...,intraop_colloid,intraop_ppf,intraop_mdz,intraop_ftn,intraop_rocu,intraop_vecu,intraop_eph,intraop_phe,intraop_epi,intraop_ca
0,1,5955,0,11542,-552,10848.0,1668,10368,-236220,627780,...,0,120,0.0,100,70,0,10,0,0,0
1,2,2487,0,15741,-1039,14921.0,1721,14621,-221160,1506840,...,0,150,0.0,0,100,0,20,0,0,0
2,3,2861,0,4394,-590,4210.0,1090,3010,-218640,40560,...,0,0,0.0,0,50,0,0,0,0,0


In [8]:
duckdb.sql("SELECT DISTINCT name FROM 'D:/akiml_data/vitaldb/lab_data.csv' ORDER BY name").df()

,name
0,alb
1,alt
2,ammo
3,aptt
4,ast
5,be
6,bun
7,ccr
8,cl
9,cr


In [9]:
duckdb.sql("""
SELECT COUNT(*) AS n, MIN(result) AS min_val, MAX(result) AS max_val, AVG(result) AS mean_val
FROM 'D:/akiml_data/vitaldb/lab_data.csv'
WHERE name = 'cre'
""").df()

,n,min_val,max_val,mean_val
0,0,NaN,NaN,NaN


In [10]:
duckdb.sql("""
SELECT COUNT(*) AS n, MIN(result) AS min_val, MAX(result) AS max_val, AVG(result) AS mean_val
FROM 'D:/akiml_data/vitaldb/lab_data.csv'
WHERE name = 'cr'
""").df()

,n,min_val,max_val,mean_val
0,37311,0.02,64.88,1.314156


In [11]:
duckdb.sql("""
SELECT l.caseid, l.dt, l.result, c.opstart, c.opend,
       (l.dt - c.opstart)/3600.0 AS hours_from_opstart
FROM 'D:/akiml_data/vitaldb/lab_data.csv' l
JOIN 'D:/akiml_data/vitaldb/clinical_data.csv' c ON l.caseid = c.caseid
WHERE l.name = 'cr'
LIMIT 10
""").df()

,caseid,dt,result,opstart,opend,hours_from_opstart
0,1,594470,0.72,1668,10368,164.667222
1,1,399575,0.75,1668,10368,110.529722
2,1,12614,0.88,1668,10368,3.040556
3,1,137855,0.91,1668,10368,37.829722
4,2,502448,0.71,1721,14621,139.090833
5,2,326060,0.76,1721,14621,90.094167
6,2,240320,0.77,1721,14621,66.277500
7,2,5857898,0.82,1721,14621,1626.715833
8,2,5776336,0.98,1721,14621,1604.059722
9,2,152248,1.02,1721,14621,41.813056


In [12]:
duckdb.sql("""
WITH cr AS (
  SELECT l.caseid, l.dt, l.result AS cr_val, c.opstart, c.opend
  FROM 'D:/akiml_data/vitaldb/lab_data.csv' l
  JOIN 'D:/akiml_data/vitaldb/clinical_data.csv' c ON l.caseid = c.caseid
  WHERE l.name = 'cr' AND l.result BETWEEN 0.2 AND 25
),
base AS (
  SELECT caseid, cr_val AS baseline_cr
  FROM (
    SELECT caseid, dt, cr_val,
           ROW_NUMBER() OVER (PARTITION BY caseid ORDER BY dt DESC) AS rn
    FROM cr
    WHERE dt <= opstart AND dt >= opstart - 7*86400
  ) WHERE rn = 1
),
post AS (
  SELECT caseid, MAX(cr_val) AS postop_cr_max
  FROM cr
  WHERE dt >= opend AND dt <= opend + 7*86400
  GROUP BY caseid
)
SELECT
  COUNT(*) AS n_with_baseline,
  COUNT(post.caseid) AS n_with_postop_cr,
  SUM(CASE WHEN postop_cr_max >= GREATEST(baseline_cr*1.5, baseline_cr+0.3) THEN 1 ELSE 0 END) AS n_aki
FROM base LEFT JOIN post USING (caseid)
""").df()

,n_with_baseline,n_with_postop_cr,n_aki
0,3741,3103,105.0


In [13]:
duckdb.sql("""
SELECT
  COUNT(*) AS n,
  ROUND(MEDIAN(baseline_cr),2) AS median_base,
  ROUND(QUANTILE_CONT(baseline_cr, 0.05),2) AS p05,
  ROUND(QUANTILE_CONT(baseline_cr, 0.25),2) AS p25
FROM (
  SELECT l.caseid, l.result AS baseline_cr,
         ROW_NUMBER() OVER (PARTITION BY l.caseid ORDER BY l.dt DESC) AS rn
  FROM 'D:/akiml_data/vitaldb/lab_data.csv' l
  JOIN 'D:/akiml_data/vitaldb/clinical_data.csv' c ON l.caseid = c.caseid
  WHERE l.name='cr' AND l.result BETWEEN 0.2 AND 25
    AND l.dt <= c.opstart AND l.dt >= c.opstart - 7*86400
) WHERE rn = 1
""").df()

InternalException: INTERNAL Error: Failed to cast expression to type - expression type mismatch
This error signals an assertion failure within DuckDB. This usually occurs due to unexpected conditions or errors in the program's logic.
For more information, see https://duckdb.org/docs/stable/dev/internal_errors

In [ ]:
duckdb.sql("""
WITH cr AS (
  SELECT l.caseid, l.dt, l.result AS cr_val, c.opstart, c.opend
  FROM 'D:/akiml_data/vitaldb/lab_data.csv' l
  JOIN 'D:/akiml_data/vitaldb/clinical_data.csv' c ON l.caseid = c.caseid
  WHERE l.name='cr' AND l.result BETWEEN 0.2 AND 25
),
base AS (
  SELECT caseid, cr_val AS baseline_cr
  FROM (SELECT caseid, dt, cr_val,
        ROW_NUMBER() OVER (PARTITION BY caseid ORDER BY dt DESC) rn
        FROM cr WHERE dt <= opstart AND dt >= opstart - 7*86400)
  WHERE rn = 1
),
flag AS (
  SELECT cr.caseid,
    MAX(CASE WHEN cr.dt >= cr.opend AND cr.dt <= cr.opend + 2*86400
              AND cr_val >= base.baseline_cr + 0.3 THEN 1 ELSE 0 END) AS crit_48h,
    MAX(CASE WHEN cr.dt >= cr.opend AND cr.dt <= cr.opend + 7*86400
              AND cr_val >= base.baseline_cr * 1.5 THEN 1 ELSE 0 END) AS crit_7d
  FROM cr JOIN base ON cr.caseid = base.caseid
  GROUP BY cr.caseid
)
SELECT
  COUNT(*) AS n_total,
  SUM(crit_48h) AS n_aki_48h,
  SUM(crit_7d) AS n_aki_7d,
  SUM(CASE WHEN crit_48h=1 OR crit_7d=1 THEN 1 ELSE 0 END) AS n_aki_kdigo
FROM base LEFT JOIN flag USING (caseid)
""").df()

In [ ]:
duckdb.sql("""
SELECT CASE WHEN baseline_cr < 0.5 THEN 'a.<0.5'
            WHEN baseline_cr < 0.7 THEN 'b.0.5-0.7'
            WHEN baseline_cr < 1.0 THEN 'c.0.7-1.0'
            WHEN baseline_cr < 1.5 THEN 'd.1.0-1.5'
            ELSE 'e.>=1.5' END AS base_grp,
       COUNT(*) AS n
FROM (
  SELECT l.caseid, l.result AS baseline_cr,
         ROW_NUMBER() OVER (PARTITION BY l.caseid ORDER BY l.dt DESC) AS rn
  FROM 'D:/akiml_data/vitaldb/lab_data.csv' l
  JOIN 'D:/akiml_data/vitaldb/clinical_data.csv' c ON l.caseid = c.caseid
  WHERE l.name='cr' AND l.result BETWEEN 0.2 AND 25
    AND l.dt <= c.opstart AND l.dt >= c.opstart - 7*86400
) WHERE rn = 1
GROUP BY 1 ORDER BY 1
""").df()

In [14]:
base = duckdb.sql("""
WITH cr AS (
  SELECT l.caseid, l.dt, l.result AS cr_val, c.opstart
  FROM 'D:/akiml_data/vitaldb/lab_data.csv' l
  JOIN 'D:/akiml_data/vitaldb/clinical_data.csv' c ON l.caseid = c.caseid
  WHERE l.name='cr' AND l.result BETWEEN 0.2 AND 25
)
SELECT caseid, cr_val AS baseline_cr
FROM (
  SELECT caseid, dt, cr_val,
         ROW_NUMBER() OVER (PARTITION BY caseid ORDER BY dt DESC) AS rn
  FROM cr
  WHERE dt <= opstart AND dt >= opstart - 7*86400
) WHERE rn = 1
""").df()

InternalException: INTERNAL Error: Failed to cast expression to type - expression type mismatch
This error signals an assertion failure within DuckDB. This usually occurs due to unexpected conditions or errors in the program's logic.
For more information, see https://duckdb.org/docs/stable/dev/internal_errors

In [15]:
import pandas as pd

lab = pd.read_csv('D:/akiml_data/vitaldb/lab_data.csv')
clin = pd.read_csv('D:/akiml_data/vitaldb/clinical_data.csv')

cr = lab[lab['name'] == 'cr'].merge(
    clin[['caseid','opstart','opend']], on='caseid', how='inner')
cr = cr[(cr['result'] >= 0.2) & (cr['result'] <= 25)].copy()

pre = cr[(cr['dt'] <= cr['opstart']) & (cr['dt'] >= cr['opstart'] - 7*86400)]
base = (pre.sort_values('dt', ascending=False)
            .drop_duplicates('caseid')[['caseid','result']]
            .rename(columns={'result':'baseline_cr'}))

base['bucket'] = pd.cut(base['baseline_cr'], bins=[0,0.5,0.7,1.0,1.5,99],
                        labels=['<0.5','0.5-0.7','0.7-1.0','1.0-1.5','>=1.5'],
                        right=False)
base['bucket'].value_counts().sort_index()

bucket
<0.5        128
0.5-0.7    1026
0.7-1.0    1708
1.0-1.5     557
>=1.5       322
Name: count, dtype: int64

In [16]:
post = cr[(cr['dt'] >= cr['opend']) & (cr['dt'] <= cr['opend'] + 7*86400)]
postmax = post.groupby('caseid')['result'].max().rename('postop_cr_max')

df = base.merge(postmax, on='caseid', how='left')
df['aki_48h'] = ((df['postop_cr_max'] >= df['baseline_cr'] + 0.3)).astype(float)
df['aki_7d']  = ((df['postop_cr_max'] >= df['baseline_cr'] * 1.5)).astype(float)
df['aki'] = ((df['aki_48h'] == 1) | (df['aki_7d'] == 1)).astype(int)

print(len(df), df['aki'].sum(), round(df['aki'].mean()*100, 1), '%')

3741 216 5.8 %


In [17]:
post48 = cr[(cr['dt'] >= cr['opend']) & (cr['dt'] <= cr['opend'] + 2*86400)]
max48 = post48.groupby('caseid')['result'].max().rename('cr_max_48h')

post7 = cr[(cr['dt'] >= cr['opend']) & (cr['dt'] <= cr['opend'] + 7*86400)]
max7 = post7.groupby('caseid')['result'].max().rename('cr_max_7d')

df = (base.merge(max48, on='caseid', how='left')
          .merge(max7, on='caseid', how='left'))
df['aki'] = ((df['cr_max_48h'] >= df['baseline_cr'] + 0.3) |
             (df['cr_max_7d'] >= df['baseline_cr'] * 1.5)).astype(int)
print(len(df), df['aki'].sum(), round(100*df['aki'].mean(), 1), '%')

3741 195 5.2 %


In [18]:
duckdb.sql("""
WITH cr AS (
  SELECT l.caseid, l.dt, l.result AS cr_val, c.opstart, c.opend
  FROM 'D:/akiml_data/vitaldb/lab_data.csv' l
  JOIN 'D:/akiml_data/vitaldb/clinical_data.csv' c ON l.caseid = c.caseid
  WHERE l.name='cr' AND l.result BETWEEN 0.2 AND 25
),
base AS (
  SELECT caseid, cr_val AS baseline_cr FROM (
    SELECT caseid, dt, cr_val,
           ROW_NUMBER() OVER (PARTITION BY caseid ORDER BY dt DESC) rn
    FROM cr WHERE dt <= opstart AND dt >= opstart - 7*86400
  ) WHERE rn = 1
),
post AS (
  SELECT caseid, MAX(cr_val) AS postop_max, COUNT(*) AS n_postop_meas
  FROM cr WHERE dt >= opend AND dt <= opend + 7*86400
  GROUP BY caseid
)
SELECT b.caseid,
       ROUND(b.baseline_cr, 2) AS baseline_cr,
       ROUND(p.postop_max, 2) AS postop_cr_max,
       p.n_postop_meas,
       CASE WHEN p.postop_max >= b.baseline_cr * 1.5 THEN 'AKI_1.5x' ELSE 'no' END AS flag
FROM base b
JOIN post p USING (caseid)
ORDER BY p.postop_max DESC
LIMIT 20
""").df()

,caseid,baseline_cr,postop_cr_max,n_postop_meas,flag
0,988,0.72,22.54,9,AKI_1.5x
1,4862,21.45,22.45,1,no
2,4708,21.92,20.80,5,no
3,4835,17.67,18.13,8,no
4,4650,19.99,17.48,8,no
5,5669,18.10,16.64,8,no
6,309,13.86,16.63,4,no
7,3323,16.44,16.06,1,no
8,1548,17.51,15.94,8,no
9,58,17.05,15.83,9,no


In [19]:
base = base[base['baseline_cr'] < 4.0].copy()   # 排除CKD4-5（相当于eGFR<30）

In [20]:
df = (base.merge(max48, on='caseid', how='left')
          .merge(max7, on='caseid', how='left'))
df['aki'] = ((df['cr_max_48h'] >= df['baseline_cr'] + 0.3) |
             (df['cr_max_7d'] >= df['baseline_cr'] * 1.5)).astype(int)
print(len(df), df['aki'].sum(), round(100*df['aki'].mean(), 1), '%')

3534 155 4.4 %


In [21]:
cr[cr['caseid'] == 988].sort_values('dt')[['dt','result','opstart','opend']]

,dt,result,opstart,opend
5671,-321711,0.72,2224,14224
5676,69720,2.03,2224,14224
5678,91432,22.54,2224,14224
5677,92755,2.37,2224,14224
5675,119960,1.96,2224,14224
5673,156806,1.05,2224,14224
5674,162887,1.57,2224,14224
5670,243569,0.71,2224,14224
5669,342883,0.59,2224,14224
5672,415649,0.73,2224,14224


In [22]:
crb = cr.merge(base[['caseid','baseline_cr']], on='caseid', how='inner')
p48 = crb[(crb['dt'] >= crb['opend']) & (crb['dt'] <= crb['opend'] + 2*86400)]
p7  = crb[(crb['dt'] >= crb['opend']) & (crb['dt'] <= crb['opend'] + 7*86400)]
n48 = p48[p48['result'] >= p48['baseline_cr'] + 0.3].groupby('caseid').size()
n7  = p7[p7['result'] >= p7['baseline_cr'] * 1.5].groupby('caseid').size()

df2 = base[['caseid','baseline_cr']].copy()
df2['n_elev48'] = df2['caseid'].map(n48).fillna(0).astype(int)
df2['n_elev7']  = df2['caseid'].map(n7).fillna(0).astype(int)
df2['aki_confirmed'] = ((df2['n_elev48'] >= 2) | (df2['n_elev7'] >= 2)).astype(int)
print(len(df2), df2['aki_confirmed'].sum(), round(100*df2['aki_confirmed'].mean(),1), '%')

3534 87 2.5 %


In [23]:
p7.groupby('caseid').size().describe()

count    2924.000000
mean        4.282832
std         3.445142
min         1.000000
25%         2.000000
50%         4.000000
75%         5.000000
max        39.000000
dtype: float64